In [302]:
from os import listdir
import numpy as np
from numpy import asarray
from numpy import save
import tensorflow as tf
from sklearn.model_selection import train_test_split

#Deshabilitar la GPU:
#tf.config.set_visible_devices([], 'GPU')
from tensorflow import keras
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

folders = listdir('./yoga')

photos =  []
labels = []

In [303]:
for idx,folder in enumerate(folders):
    for file in listdir('./yoga/'+folder):
        photo = load_img('./yoga/'+folder+'/' + file) 
        photo = img_to_array(photo)
        photos.append(photo)
        labels.append(float(idx))
        del photo
    # print(idx)

In [304]:
photos = np.array(photos)
labels = np.array(labels)

In [305]:
photos.shape

(570, 16, 16, 3)

In [306]:
photos.shape

(570, 16, 16, 3)

In [307]:
X = photos
y = labels

In [308]:
from sklearn.preprocessing import StandardScaler

# X = X/255.0

X = X.reshape(X.shape[0], -1)

escalador = StandardScaler()
X = escalador.fit_transform(X)

In [309]:
X = X.reshape(570, 16, 16 ,3)

In [310]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)

In [311]:
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPool2D
from tensorflow.keras.layers import Flatten

model = keras.models.Sequential()
model.add(Conv2D(16,(3,3), activation='relu', input_shape=(16, 16, 3)))
model.add(MaxPool2D(2,2))
# model.add(Conv2D(64,(3,3), activation='relu'))
# model.add(MaxPool2D(2,2))
# model.add(Conv2D(128,(3,3), activation='relu'))
# model.add(MaxPool2D(2,2))
model.add(Flatten())
model.add(keras.layers.Dense(512,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(128,activation='relu',kernel_initializer='he_normal'))
# model.add(keras.layers.Dense(100,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(38,activation='softmax',kernel_initializer='glorot_normal'))

/home/ciabd14/anaconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [312]:
model.compile(loss='sparse_categorical_crossentropy', optimizer = keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])

early_stopping_cb = keras.callbacks.EarlyStopping(patience=10,restore_best_weights=True)

history = model.fit(X_train, y_train, epochs=10000,validation_split = 0.1,callbacks=[early_stopping_cb],batch_size=256)

Epoch 1/10000
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step - accuracy: 0.0217 - loss: 3.9045 - val_accuracy: 0.0000e+00 - val_loss: 3.9167
Epoch 2/10000
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.0282 - loss: 3.7432 - val_accuracy: 0.0385 - val_loss: 3.7741
Epoch 3/10000
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.0521 - loss: 3.6101 - val_accuracy: 0.0385 - val_loss: 3.6535
Epoch 4/10000
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.0781 - loss: 3.4941 - val_accuracy: 0.0577 - val_loss: 3.5508
Epoch 5/10000
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.0998 - loss: 3.3916 - val_accuracy: 0.0769 - val_loss: 3.4583
Epoch 6/10000
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.1215 - loss: 3.2998 - val_accuracy: 0.0962 - val_loss: 3.3752
Epoch 7/10000
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.1540 - loss: 3.2144 - val_accuracy: 0.1154 - val_loss: 3.2984
Epoch 8/10000
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.1887 - loss: 3.1326 - val_accurac

In [313]:
model.evaluate(X_test,y_test)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9123 - loss: 0.5386


[0.5386213064193726, 0.9122806787490845]

## RANDOM FOREST

In [314]:
X_train_reshape = X_train.reshape(X_train.shape[0], -1)
X_test_reshape = X_test.reshape(X_test.shape[0], -1)

In [315]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(n_estimators=600, n_jobs=1, random_state=42)
rf_clf.fit(X_train_reshape, y_train)

RandomForestClassifier(n_estimators=600, n_jobs=1, random_state=42)

In [316]:
y_pred = rf_clf.predict(X_test_reshape)

In [317]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión del árbol de decisión: {accuracy:.4f} ({accuracy*100:.2f}%)")

Precisión del árbol de decisión: 0.8772 (87.72%)
